In [1]:
from pathlib import Path
import cv2
import numpy as np
from mmdet.apis import inference_detector, init_detector

In [2]:
config_path = Path(r"C:\dev\projects\CV_counting_bags\configs\rtmdet_tiny_bag.py")
ckpt_path = Path(r"C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth")
video_path = Path(r"C:\dev\projects\CV_counting_bags\input.mp4")
out_video_path = Path(r"C:\dev\projects\CV_counting_bags\output_video\output.mp4")

In [3]:
model = init_detector(
    str(config_path),
    str(ckpt_path),
    device = "cuda:0"
)

Loads checkpoint by local backend from path: C:\dev\projects\CV_counting_bags\work_dirs\rtmdet_tiny_bag\best_coco_bbox_mAP_epoch_27.pth


In [4]:
conveyor_roi = np.array([
    [0, 360],
    [310, 28],
    [484, 65],
    [315, 360],    
], dtype=np.int32)

In [5]:
def interpolate(p1, p2, coeff):
    x = p1[0] + (p2[0] - p1[0]) * coeff
    y = p1[1] + (p2[1] - p1[1]) * coeff

    return int(x), int(y)

In [41]:
def get_forward_vector(roi):
    top_center = np.array([(roi[1][0] + roi[2][0]) / 2, (roi[1][1] + roi[2][1]) / 2], dtype = float)

    bottom_center = np.array([(roi[0][0] + roi[3][0]) / 2, (roi[0][1] + roi[3][1]) / 2], dtype = float)

    vector = top_center - bottom_center
    return vector / np.linalg.norm(vector)

In [31]:
forward_vector = get_forward_vector(conveyor_roi)

In [6]:
line_pos_coeff = 0.35

left_point = interpolate(conveyor_roi[1], conveyor_roi[0], line_pos_coeff)
right_point = interpolate(conveyor_roi[2], conveyor_roi[3], line_pos_coeff)

counting_line = (left_point, right_point)

In [7]:
def bbox_iou(bbox1, bbox2):
    x1 = max(bbox1[0], bbox2[0])
    y1 = max(bbox1[1], bbox2[1])
    x2 = min(bbox1[2], bbox2[2])
    y2 = min(bbox1[3], bbox2[3])

    inter_width = max(0, x2 - x1)
    inter_height = max(0, y2 - y1)

    area1 = max(0, bbox1[2] - bbox1[0]) * max(0, bbox1[3] - bbox1[1])
    area2 = max(0, bbox2[2] - bbox2[0]) * max(0, bbox2[3] - bbox2[1])

    intersection = inter_width * inter_height
    union = area1 + area2 - intersection
    
    if union == 0:
        return 0.0

    return intersection / union

In [43]:
class Track:
    def __init__(self, track_id, bbox, score):
        self.id = track_id
        self.bbox = np.array(bbox, dtype = float)
        self.score = float(score)

        self.hits = 1
        self.missed = 0
        self.age = 1
        self.confirmed = False
        self.history = [self.center()]

        self.last_side = None
        self.moving = False
    
    def center(self):
        x1, y1, x2, y2 = self.bbox

        return(float((x1 + x2) / 2), float((y1 + y2) / 2))

    def update(self, bbox, score):
        self.bbox = np.array(bbox, dtype = float)
        self.score = float(score)

        self.hits += 1
        self.missed = 0
        self.age += 1
        self.history.append(self.center())

    def mark_missed(self):
        self.missed += 1
        self.age += 1

In [49]:
class IoUTracker:
    def __init__(self, iou_threshold = 0.3, max_missed = 3, min_hits = 3):
        self.iou_threshold = iou_threshold
        self.max_missed = max_missed
        self.min_hits = min_hits

        self.tracks = []
        self.next_id = 1

    def create_track(self, detection):
        track = Track(track_id = self.next_id, bbox = detection['bbox'], score = detection['score'])
        
        self.next_id += 1
        self.tracks.append(track)

    def update(self, detections):
        if len(self.tracks) == 0:
            for detection in detections:
                self.create_track(detection)

            self.update_accepted()
            return self.tracks

        if len(detections) == 0:
            for track in self.tracks:
                track.mark_missed()

            self.remove_dead_tracks()
            self.update_accepted()
            return self.tracks

        iou_matrix = np.zeros((len(self.tracks), len(detections)), dtype = float)

        for track_index, track in enumerate(self.tracks):
            for detection_index, detection in enumerate(detections):
                iou_matrix[track_index, detection_index] = bbox_iou(track.bbox, detection['bbox'])

        matched_tracks = set()
        matched_detections = set()

        while True:
            track_index, detection_index = np.unravel_index(np.argmax(iou_matrix), iou_matrix.shape)

            best_iou = iou_matrix[track_index, detection_index]
            if best_iou < self.iou_threshold:
                break

            track = self.tracks[track_index]
            detection = detections[detection_index]

            track.update(detection['bbox'], detection['score'])

            matched_tracks.add(track_index)
            matched_detections.add(detection_index)

            iou_matrix[track_index, :] = -1
            iou_matrix[:, detection_index] = -1

        for track_index, track in enumerate(self.tracks):
            if track_index not in matched_tracks:
                track.mark_missed()

        for detection_index, detection in enumerate(detections):
            if detection_index not in matched_detections:
                self.create_track(detection)

        self.remove_dead_tracks()
        self.update_accepted()

        return self.tracks

    def remove_dead_tracks(self):
        self.tracks = [track for track in self.tracks if track.missed <= self.max_missed]


    def update_accepted(self):
        for track in self.tracks:
            if track.hits >= self.min_hits:
                track.confirmed = True            

In [45]:
class AnomalyMonitor:
    def __init__(self, motion_window = 10, movement_threshold = 0.2, reverse_confirm_sec = 0.5, stop_confirm_sec = 2.0):
        self.anomalies = []
        self.active = {}

        self.motion_window = motion_window
        self.movement_threshold = movement_threshold

        self.reverse_confirm_sec = reverse_confirm_sec
        self.stop_confirm_sec = stop_confirm_sec

        self.reverse_frames = 0
        self.stopped_frames = 0


    def get_motion_state(self, track, forward_vector):
        window = self.motion_window

        if len(track.history) < window + 1:
            return None

        prev = np.array(track.history[-window - 1], dtype = float)
        curr = np.array(track.history[-1], dtype = float)

        movement = curr - prev
        speed = (np.linalg.norm(movement) / window)
        projection = (np.dot(movement, forward_vector) / window)

        if speed < self.movement_threshold:
            return "stationary"

        if projection > 0:
            return "forward"

        return "reverse"

    def activate(self, name, frame_index, fps):
        if name in self.active:
            return

        anomaly = {
            "type": name,
            "start_frame": frame_index,
            "start_time": frame_index / fps,
            "end_frame": None,
            "end_time": None
        }

        self.active[name] = anomaly
        self.anomalies.append(anomaly)    

    def resolve(self, name, frame_index, fps):
        if name not in self.active:
            return

        anomaly = self.active[name]

        anomaly['end_frame'] = frame_index
        anomaly['end_time'] = frame_index / fps

        del self.active[name]

    def update_motion(self, tracks, forward_vector, frame_index, fps):
        states = []

        for track in tracks:
            if not track.confirmed:
                continue

            state = self.get_motion_state(track, forward_vector)

            if state is None:
                continue

            if state in ("forward", "reverse"):
                track.moving = True

            if state == "stationary" and not track.moving:
                continue

            states.append(state)

        if len(states) == 0:
            return

        forward_count = states.count("forward")
        reverse_count = states.count("reverse")
        stationary_count = states.count("stationary")

        total = len(states)

        is_reverse = (reverse_count > forward_count and reverse_count > stationary_count)

        if is_reverse:
            self.reverse_frames += 1
        else:
            self.reverse_frames = 0

        reverse_confirm = int(self.reverse_confirm_sec * fps)

        if self.reverse_frames >= reverse_confirm:
            self.activate("conveyor reverse", frame_index, fps)

        is_forward = (forward_count > reverse_count and forward_count > stationary_count)

        if ("conveyor reverse" in self.active and is_forward):
            self.resolve("conveyor reverse", frame_index, fps)

        is_stopped = (stationary_count == total)

        if is_stopped:
            self.stopped_frames += 1
        else:
            self.stopped_frames = 0

        stop_confirm = int(self.stop_confirm_sec * fps)

        if self.stopped_frames >= stop_confirm:
            self.activate("conveyor stopped", frame_index, fps)

        is_moving = (forward_count > 0 or reverse_count > 0)

        if ("conveyor stopped" in self.active and is_moving):
            self.resolve("conveyor stopped", frame_index, fps)

In [36]:
class LineCounter:
    def __init__(self, line):
        self.line = line
        self.count = 0

    def point_side(self, point):
        x, y = point
        (x1, y1), (x2, y2) = self.line

        value = ((x2 - x1) * (y - y1) - (y2 - y1) * (x - x1))

        if value > 0:
            return 1
        elif value < 0:
            return -1
        else:
            return 0

    def update(self, track):
        curr_side = self.point_side(track.center())

        if curr_side == 0:
            return None
            
        if track.last_side is None:
            track.last_side = curr_side
            return None

        if track.last_side == 1 and curr_side == -1:
            self.count += 1

        if track.last_side == -1 and curr_side == 1:
            self.count -= 1

        track.last_side = curr_side

In [33]:
def inside_roi(obj, roi):
    obj_center = (int((obj[0] + obj[2]) / 2), int((obj[1] + obj[3]) / 2))
    result = cv2.pointPolygonTest(roi, obj_center, measureDist = False)

    if result >= 0:
        return True

In [13]:
def draw_detection(frame, result, roi, threshold = 0.36):
    image = frame.copy()

    pred = result.pred_instances
    bboxes = pred.bboxes.detach().cpu().numpy()
    scores = pred.scores.detach().cpu().numpy()

    detections_sum = 0
    track_detections = []

    for bbox, score in zip(bboxes, scores):
        if score < threshold:
            continue

        if inside_roi(bbox, roi):
            track_detections.append({
                "bbox": bbox,
                "score": score
            })

        detections_sum += 1

        x1, y1, x2, y2 = bbox.astype(int)

        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        label = f"bag {score:.2f}"

        cv2.putText(image, label, (x1, max(y1 - 7, 20)), 
            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    cv2.putText(image, f"Detections: {detections_sum}",
        (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    cv2.putText(image, f"Tracking candidates: {len(track_detections)}",
        (20, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)

    #cv2.line(image, counting_line[0],
     #   counting_line[1], (0, 255, 255), 2)

    return image, track_detections

In [50]:
def process_video(video_path, out_path, model, threshold = 0.36, max_frames = None):
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        raise RuntimeError("failed to open video:", video_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frames_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    print(f"FPS: {fps}")
    print(f"Res: {width}x{height}")
    print(f"Frames: {frames_count}")

    out_path = Path(out_path)
    out_path.parent.mkdir(parents = True, exist_ok = True)
    
    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(str(out_path), fourcc, fps, (width, height))

    if not writer.isOpened():
        cap.release()
        raise RuntimeError("failed to create video", out_path)
        
    tracker = IoUTracker(iou_threshold = 0.3, max_missed = 3, min_hits = 3)
    
    frame_index = 0
    counter = LineCounter(counting_line)
    monitor = AnomalyMonitor()
    
    while True:
        ret, frame = cap.read()

        if not ret:
            break

        if max_frames is not None and frame_index >= max_frames:
            break

        result = inference_detector(model, frame)
        annotation, track_detections = draw_detection(frame, result, conveyor_roi, threshold = threshold)
        tracks = tracker.update(track_detections)
        monitor.update_motion(tracks, forward_vector, frame_index, fps)

        for track in tracks:
            if not track.confirmed:
                continue
            
            x1, y1, x2, y2 = track.bbox.astype(int)

            counter.update(track)
            
            cv2.putText(annotation, f"ID {track.id}", (x1, y2 + 20),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        cv2.putText(annotation, f"Bags count: {counter.count}",
            (20, 110), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 255), 2)

        for anomaly_name in monitor.active:
            cv2.putText(annotation, f"ANOMALY: {anomaly_name}",
                (360, 300), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)

        writer.write(annotation)
        frame_index += 1

    cap.release()
    writer.release()
    print("done")
    
    return {
        "count": counter.count,
        "anomalies": monitor.anomalies
    }

In [46]:
process_video(video_path, out_path = out_video_path, model = model)

FPS: 25.0
Res: 640x360
Frames: 14999
done


{'count': 125,
 'anomalies': [{'type': 'conveyor reverse',
   'start_frame': 2730,
   'start_time': 109.2,
   'end_frame': 3486,
   'end_time': 139.44},
  {'type': 'conveyor stopped',
   'start_frame': 3550,
   'start_time': 142.0,
   'end_frame': 4366,
   'end_time': 174.64},
  {'type': 'conveyor reverse',
   'start_frame': 4377,
   'start_time': 175.08,
   'end_frame': 4410,
   'end_time': 176.4}]}